<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-25</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>청킹 전략(Chunking Strategy)</strong>에 대해 학습합니다.
>긴 문서를 효과적으로 분할하는 청킹 기법과 파라미터의 영향을 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [4]:
# TODO 0: 환경변수를 로드하고, 이전 실습에서 만든 all_documents를 준비해봅시다.

import os, glob
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader

load_dotenv()

# data/ 폴더의 모든 PDF 로드
all_documents = []
for pdf_path in sorted(glob.glob("data/*.pdf")):
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

print(f"PDF {len(glob.glob('data/*.pdf'))}개 → {len(all_documents)}페이지 로드 완료")

PDF 8개 → 8페이지 로드 완료


</br>

# 청킹 (Chunking)
> 긴 문서를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">적절한 크기의 조각</mark>으로 분할하여 검색 정확도와 LLM 처리 효율을 높입니다.

> 긴 문서를 그대로 LLM에 넣지 못하는 데에는 두 가지 이유가 있습니다.</br></br>
> 첫째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM 토큰 제한</mark>으로 LLM은 한 번에 처리할 수 있는 텍스트 길이(컨텍스트 윈도우)에 상한이 있어서 수백 페이지짜리 PDF를 통째로 프롬프트에 넣는 것은 불가능하거나, 가능하더라도 비용이 매우 큽니다.</br></br>
> 둘째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색 정확도 향상</mark>을 위해 문서를 작은 단위로 쪼개두면, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">질문과 가장 관련 있는 청크만 선별</mark>하여 LLM에 전달할 수 있습니다.</br></br>
 > 즉, 청킹은 토큰 제한 안에서 검색 품질을 최대화하기 위한 필수 전처리 단계입니다. 청크가 너무 작으면 문맥이 끊기고, 너무 크면 LLM에 불필요한 정보까지 전달됩니다.</br>

</br>

## RecursiveCharacterTextSplitter
> LangChain의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재귀적 문자 분할기</mark>로, 구분자 우선순위에 따라 의미 단위로 분할합니다.

In [5]:
# TODO 1: 재귀적 텍스트 분할기를 chunk_size=500, chunk_overlap=50, 구분자 우선순위 ["\n\n", "\n", ".", " "]로 초기화하고, 문서를 분할하여 청크 개수와 크기 통계를 출력해봅시다.

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 청크 최대 크기
    chunk_overlap=50,    # 청크 간 겹침
    separators=["\n\n", "\n", ".", " "]  # 분할 우선순위
)

chunks = splitter.split_documents(all_documents)
print(f"원본: {len(all_documents)}개 → 청크: {len(chunks)}개")

# 청크 크기 확인
sizes = [len(c.page_content) for c in chunks]
print(f"청크 크기 — 평균: {sum(sizes)/len(sizes):.0f}, 최소: {min(sizes)}, 최대: {max(sizes)}")

원본: 8개 → 청크: 33개
청크 크기 — 평균: 441, 최소: 150, 최대: 499


</br>

## 주요 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th>영향</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>chunk_size</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청크 최대 문자 수</mark></td><td>작을수록 정밀, 클수록 문맥 보존</td></tr>
    <tr><td style="text-align:center"><code>chunk_overlap</code></td><td>인접 청크 간 겹침</td><td>문맥 단절 방지</td></tr>
    <tr><td style="text-align:center"><code>separators</code></td><td>분할 기준 문자</td><td>의미 단위 보존</td></tr>
  </tbody>
</table>

</br>

## 청크 크기의 트레이드오프

In [6]:
# TODO 2: chunk_size를 200, 500, 1000으로 변경하며 재귀적 텍스트 분할기(chunk_overlap=50)로 문서를 분할하고, 각 설정별 청크 수를 비교해봅시다.

for size in [200, 500, 1000]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    c = s.split_documents(all_documents)
    print(f"chunk_size={size:4d} → 청크 수: {len(c):3d}개")

chunk_size= 200 → 청크 수:  94개
chunk_size= 500 → 청크 수:  33개
chunk_size=1000 → 청크 수:  18개


</br>

## 청킹 + 임베딩 → 검색 결과 확인
> 청킹된 문서를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벡터DB에 저장</mark>하고, 실제 검색을 수행하여 청킹 전후의 차이를 확인합니다.

In [7]:
# TODO 3: 청킹된 문서를 임베딩하여 벡터DB에 저장하고, 검색기를 생성해봅시다.

from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = UpstageEmbeddings(model="embedding-query")

# 청킹된 문서로 벡터DB 생성
chunks = splitter.split_documents(all_documents)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"청크 {len(chunks)}개 → 벡터DB 저장 완료")
print(f"검색기 생성 완료 (top-k=3)")

청크 33개 → 벡터DB 저장 완료
검색기 생성 완료 (top-k=3)


In [8]:
# TODO 4: 의미 검색을 수행하여 청킹된 결과를 확인해봅시다.

queries = [
    "늦게 도착하면 보상받을 수 있나요?",
    "처음 가입하면 뭐가 좋아요?",
    "절판된 책은 어떻게 되나요?"
]

for query in queries:
    results = retriever.invoke(query)
    print(f'\n질문: "{query}"')
    for j, doc in enumerate(results, 1):
        source = doc.metadata.get("source_file", "?")
        print(f"  [{j}] ({source}) {doc.page_content[:80]}...")


질문: "늦게 도착하면 보상받을 수 있나요?"
  [1] (배송지연 보상제도_서비스 혜택 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 ...
  [2] (배송지연 보상제도_서비스 혜택 - 예스24.pdf) 비회원 제외
아침/당일/하루 배송 대상이 아닌 경우 제외
제휴사 주문 제외
부재 등의 고객님의 사유로 배송 지연된 경우 제외
설날, 추석 등 명...
  [3] (도서 품절 보상제도 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 배송하는 도서(잡지, 중고도서, 직수입 도서 제외)가 일시품절, 품절, 절판, 판매금지, 미출간 상태로 ...

질문: "처음 가입하면 뭐가 좋아요?"
  [1] (신규 회원 혜택_서비스 혜택 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
2만원 이상 주문 시 사용
최대 1만원까지 사용가능
5만원 이상 주문 시 사용
뮤지컬, 연극만 사용가능
크레마클럽 가...
  [2] (신규 회원 혜택_서비스 혜택 - 예스24.pdf) 크레마클럽 X FLO 99 요금제 가입자는 이용권 등록이 불가합니다.
상품권은 본인인증 가입 후 이벤트 페이지에서 발급 받으실 수 있습니다. 
...
  [3] (신규 회원 혜택_서비스 혜택 - 예스24.pdf) 발급받기
첫 구매 감사쿠폰
할인쿠폰
2,000
회원 가입하기
회사소개
인재채용
이용약관
개인정보처리방침
청소년보호정책
도서홍보안내
광고안내
제휴...

질문: "절판된 책은 어떻게 되나요?"
  [1] (도서 품절 보상제도 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 배송하는 도서(잡지, 중고도서, 직수입 도서 제외)가 일시품절, 품절, 절판, 판매금지,

In [9]:
# TODO 5: 청킹 전(페이지 단위)과 청킹 후의 검색 결과를 비교해봅시다.

# 청킹 없이 원본 문서로 벡터DB 생성
vectorstore_raw = Chroma.from_documents(all_documents, embeddings, collection_name="raw_docs")
retriever_raw = vectorstore_raw.as_retriever(search_kwargs={"k": 3})

query = "늦게 도착하면 보상받을 수 있나요?"
print(f'질문: "{query}"')

print(f"\n--- 청킹 전 (페이지 단위, {len(all_documents)}개) ---")
for doc in retriever_raw.invoke(query):
    source = doc.metadata.get("source_file", "?")
    print(f"  ({source}) {doc.page_content[:80]}...")

print(f"\n--- 청킹 후 (chunk_size=500, {len(chunks)}개) ---")
for doc in retriever.invoke(query):
    source = doc.metadata.get("source_file", "?")
    print(f"  ({source}) {doc.page_content[:80]}...")

print(f"\n✓ 청킹 후: 더 관련성 높은 텍스트 조각을 정확히 검색!")

질문: "늦게 도착하면 보상받을 수 있나요?"

--- 청킹 전 (페이지 단위, 8개) ---
  (배송지연 보상제도_서비스 혜택 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 ...
  (도서 품절 보상제도 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 배송하는 도서(잡지, 중고도서, 직수입 도서 제외)가 일시품절, 품절, 절판, 판매금지, 미출간 상태로 ...
  (총알배송_서비스 혜택 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
전국 방방곡곡으로 총알배송 서비스는 계속 확대됩니다.
당일배송 표시상품을 주문하시면 당일에 받으실 수 있습니다.
주문...

--- 청킹 후 (chunk_size=500, 33개) ---
  (배송지연 보상제도_서비스 혜택 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 ...
  (배송지연 보상제도_서비스 혜택 - 예스24.pdf) 비회원 제외
아침/당일/하루 배송 대상이 아닌 경우 제외
제휴사 주문 제외
부재 등의 고객님의 사유로 배송 지연된 경우 제외
설날, 추석 등 명...
  (도서 품절 보상제도 - 예스24.pdf) 싸다
빠르다
믿을 수 있다
예스24에서 배송하는 도서(잡지, 중고도서, 직수입 도서 제외)가 일시품절, 품절, 절판, 판매금지, 미출간 상태로 ...

✓ 청킹 후: 더 관련성 높은 텍스트 조각을 정확히 검색!


💡청킹이 검색 품질에 미치는 영향
> 페이지 단위로 저장하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">관련 없는 내용이 함께 포함</mark>되어 검색 정확도가 떨어집니다.</br>
> 적절한 크기로 청킹하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">핵심 내용만 정확히 검색</mark>되어 RAG 파이프라인의 성능이 향상됩니다.</br>
> 단, 너무 작게 자르면 문맥이 끊기고, 너무 크면 노이즈가 포함됩니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">chunk_size</th>
      <th style="text-align:center">청크 수</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">200</td><td style="text-align:center">87개</td></tr>
    <tr><td style="text-align:center">500</td><td style="text-align:center">42개</td></tr>
    <tr><td style="text-align:center">1000</td><td style="text-align:center">23개</td></tr>
  </tbody>
</table>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">chunk_size</th>
      <th>장점</th>
      <th>단점</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">작음 (200)</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정밀한 검색</mark></td><td>문맥 손실</td></tr>
    <tr><td style="text-align:center">중간 (500)</td><td>균형 잡힌 성능</td><td>-</td></tr>
    <tr><td style="text-align:center">큼 (1000)</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">풍부한 문맥</mark></td><td>검색 정밀도 저하</td></tr>
  </tbody>
</table>

💡chunk_overlap의 역할
> overlap이 없으면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문장이 중간에 잘릴</mark> 수 있습니다.
> 일반적으로 chunk_size의 10~20%를 overlap으로 설정합니다.

💡청킹이 RAG 성능을 좌우한다
> 같은 모델, 같은 임베딩이라도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청킹 전략</mark>에 따라 검색 품질이 크게 달라집니다.
> 문서 특성에 맞는 chunk_size와 separator를 실험적으로 결정해야 합니다.